# NeuroDecode–LaBraM | Kaggle GPU full-data experiment (Python 3.13-compatible launcher)

**Goal:** Run **one complete, subject-held-out motor-imagery EEG training experiment** per Kaggle session with an immutable source commit and real, auditable outputs.

**Important update (environment fix):** Kaggle currently may use **Python 3.13** as its notebook kernel, but the pinned `torchaudio==2.5.1` CUDA 12.1 wheel is not provided for Python 3.13. This Notebook uses **uv** to create an isolated **Python 3.11 virtual environment** and runs **all model, dataset, evaluation and plotting subprocesses** with that interpreter. Your Kaggle Notebook's kernel remains unchanged; **do not run `pip install torch==2.5.1 torchaudio==2.5.1` in its Python 3.13 environment**.

**Required Kaggle settings:**
1. **Accelerator: GPU** and **Internet: On**. Restart the Kaggle session if GPU was changed.
2. Import this updated Notebook, then click **Run All**. The first run downloads packages, public EEG data and the pinned model; make sure the disk and time quota suffice.
3. After a completed experiment, **Save Version** and download `results_only_*.zip` from Outputs. Keep EEG caches and any subject-specific artifacts private and outside the public code repository.

The default performs **Pretrained / Seed 42**. For the full comparison run six separate model/seed configurations and keep `SPLIT_SEED=42` and the pinned source commit unchanged. A full Kaggle GPU run has not yet been validated in advance.


In [ ]:
# ==========================================================
# EDIT ONLY THIS CELL BETWEEN EXPERIMENT RUNS
# ==========================================================
EXPERIMENT = "pretrained"           # "pretrained" or "scratch"
SEED = 42                           # Repeat for: 42, 123, 2025
SPLIT_SEED = 42                     # NEVER change between compared runs
BATCH_SIZE = 16                     # Reduce to 8 or 4 if GPU runs out of memory
MAX_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
RUN_LABEL = ""                       # If retrying a failed run, e.g. "retry2"

# Optional: path to a previously validated cache you attached as a Kaggle Dataset.
# Example: "/kaggle/input/my-private-eeg-cache/physionet_mi_104subjects.h5"
ATTACHED_CACHE_PATH = ""

# The training code is pinned to this previously reviewed commit. Do not swap to
# GitHub main mid-campaign; doing so could invalidate cross-run comparisons.
REPOSITORY = "https://github.com/mohamad679/labram-mi-project.git"
SOURCE_COMMIT = "74e511fb9c1ac4f0543b249b571e28e811ec92c3"

assert EXPERIMENT in ("pretrained", "scratch")
assert SEED in (42, 123, 2025), "For the planned campaign, choose 42, 123, or 2025"
assert SPLIT_SEED == 42, "Fixed subject split is mandatory for comparability"
assert 1 <= BATCH_SIZE <= 64 and 1 <= MAX_EPOCHS <= 200
print(f"Selected run: {EXPERIMENT}, seed={SEED}, split_seed={SPLIT_SEED}, batch={BATCH_SIZE}")


## 1. GPU and storage preflight

Fails **before downloading the dataset** if Kaggle does not have an accessible NVIDIA GPU. The CUDA/PyTorch runtime is subsequently checked after installation.


In [ ]:
import os, sys, subprocess, shutil, json, time
from pathlib import Path

# Jupyter sets an inline-only matplotlib backend unsupported by isolated Python 3.11.
# Use a non-interactive backend for all child processes and saved scientific plots.
os.environ["MPLBACKEND"] = "Agg"

assert Path("/kaggle/working").exists(), "Run this notebook on Kaggle, not a standard local kernel."
try:
    info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          text=True, capture_output=True, check=True)
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    raise RuntimeError("Kaggle GPU not available. Settings → Accelerator → GPU, then restart the session.") from exc
print("GPU(s):\n", info.stdout.strip())
free_gb = shutil.disk_usage("/kaggle/working").free / (1024**3)
print(f"Available /kaggle/working disk space: {free_gb:.1f} GiB")
if free_gb < 8:
    raise RuntimeError("Under 8 GiB of free disk. Clear old outputs or attach a validated cache before training.")
WORK = Path("/kaggle/working")
OUTPUT_ROOT = WORK / "neurodecode_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_ROOT / f"{EXPERIMENT}_seed_{SEED}{'_' + RUN_LABEL if RUN_LABEL else ''}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Run output directory:", RUN_DIR)


## 2. Create a compatible Python 3.11 GPU environment

The earlier notebook failed on **Kaggle Python 3.13** because PyTorch 2.5.1's CUDA 12.1 wheel existed for `cp313` but **Torchaudio 2.5.1's CUDA 12.1 wheel does not**. This is a wheel compatibility issue, not a training/model problem.

The cell below bootstraps [uv](https://docs.astral.sh/uv/), installs its managed CPython 3.11 in a private virtual environment, and installs the original verified GPU library versions **there**, not in Kaggle's Python 3.13 kernel. All downstream subprocesses use the `PYTHON` variable pointing into this environment. Installation logs are saved to the run folder.

**Internet must be enabled.** When retrying after the original failed installation, rerun the entire updated Notebook from the beginning rather than retrying the old installation cell.


In [ ]:
import subprocess, sys, shutil, os
from pathlib import Path

# Keep Kaggle's notebook kernel untouched; it may be Python 3.13.
print("Notebook/kernel Python:", sys.version.split()[0])
VENV_DIR = WORK / "neurodecode_python311"
PYTHON = str(VENV_DIR / "bin" / "python")
install_log = RUN_DIR / "installation.log"

def logged_install(cmd):
    print("Installing:", " ".join(map(str, cmd)), flush=True)
    with install_log.open("a", encoding="utf-8") as log:
        log.write("\n$ " + " ".join(map(str, cmd)) + "\n")
        result = subprocess.run(list(map(str, cmd)), stdout=log,
                                stderr=subprocess.STDOUT, text=True)
    if result.returncode:
        print(install_log.read_text(encoding="utf-8", errors="replace")[-10000:])
        raise RuntimeError(f"Environment setup failed (exit {result.returncode}). "
                           f"See {install_log}; no dataset/training has started.")

# uv can download managed Python 3.11 even if /usr/bin/python3.11 is absent.
uv = shutil.which("uv")
if uv is None:
    logged_install([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                    "--no-input", "uv"])
    uv = shutil.which("uv") or str(Path(sys.executable).parent / "uv")
if not Path(uv).is_file():
    raise RuntimeError("uv executable was not found after installation.")
print("uv:", uv)

# Reuse a compatible environment within this Kaggle session if present.
if not Path(PYTHON).is_file():
    logged_install([uv, "python", "install", "3.11"])
    logged_install([uv, "venv", "--python", "3.11", "--seed", str(VENV_DIR)])
version = subprocess.check_output([PYTHON, "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
                                  text=True).strip()
if version != "3.11":
    raise RuntimeError(f"Expected isolated Python 3.11, found {version}")
print("Training/evaluation Python:", PYTHON, "(version", version + ")")

# CUDA PyTorch 2.5.1 + torchaudio 2.5.1 both have CPython 3.11 wheels.
logged_install([uv, "pip", "install", "--python", PYTHON,
                "--index-url", "https://download.pytorch.org/whl/cu121",
                "torch==2.5.1", "torchaudio==2.5.1", "torchvision==0.20.1"])
# Keep the previously CPU-validated Braindecode/MOABB/MNE stack.
logged_install([uv, "pip", "install", "--python", PYTHON,
                "braindecode[hub]==1.5.1", "moabb==1.7.2", "mne==1.13.2",
                "torch==2.5.1", "torchaudio==2.5.1", "torchvision==0.20.1",
                "safetensors>=0.4,<1", "scikit-learn>=1.3,<2", "numpy>=1.26,<3",
                "PyYAML>=6,<7", "matplotlib>=3.8,<4", "pytest>=8,<9", "h5py>=3.10,<4"])
print("Isolated Python 3.11 GPU environment ready. Full log:", install_log)


## 3. Clone immutable experiment source and verify GPU libraries

The pinned commit is intentionally **not** the moving `main` branch. This guarantees that the six planned runs use the same source code even if the GitHub repository evolves.


In [ ]:
import os, subprocess, sys, json
from pathlib import Path

# Also set this here so the verification cell can be rerun independently.
os.environ["MPLBACKEND"] = "Agg"

SOURCE_DIR = WORK / "neurodecode_source"
if not (SOURCE_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", REPOSITORY, str(SOURCE_DIR)], check=True)
subprocess.run(["git", "-C", str(SOURCE_DIR), "checkout", "--detach", SOURCE_COMMIT], check=True)
actual_sha = subprocess.check_output(["git", "-C", str(SOURCE_DIR), "rev-parse", "HEAD"], text=True).strip()
assert actual_sha == SOURCE_COMMIT
print("Reproducible source:", actual_sha)

# The probe runs in Python 3.11 -- never import pinned torch in the Kaggle kernel.
probe = """import sys, torch, torchaudio, braindecode, moabb, mne
assert sys.version_info[:2] == (3, 11)
assert torch.cuda.is_available(), 'PyTorch 3.11 CUDA is unavailable. Check Kaggle GPU accelerator and CUDA driver.'
assert torch.__version__.startswith('2.5.1'), torch.__version__
assert torchaudio.__version__.startswith('2.5.1'), torchaudio.__version__
assert braindecode.__version__ == '1.5.1', braindecode.__version__
assert moabb.__version__ == '1.7.2', moabb.__version__
print('Python:', sys.version.split()[0], 'Torch:', torch.__version__, 'Torchaudio:', torchaudio.__version__)
print('Braindecode:', braindecode.__version__, 'MOABB:', moabb.__version__, 'MNE:', mne.__version__)
print('GPU:', torch.cuda.get_device_name(0), 'GPU Memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory/2**30, 2))
"""
subprocess.run([PYTHON, "-c", probe], check=True, cwd=SOURCE_DIR)

# Fast offline tests. Detect incompatible code before the large public-data download.
with (RUN_DIR / "unit_tests.log").open("w", encoding="utf-8") as log:
    result = subprocess.run([PYTHON, "-m", "pytest", "-q"], cwd=SOURCE_DIR,
                            text=True, stdout=log, stderr=subprocess.STDOUT)
print((RUN_DIR / "unit_tests.log").read_text(encoding="utf-8")[-3500:])
if result.returncode:
    raise RuntimeError("Repository offline tests failed in Python 3.11; abort GPU training.")


## 4. Attach verified, locally generated HDF5 cache (required)

**Do not bulk-download EEG in Kaggle.** Use `python -m scripts.download_physionet_local --output physionet_mi_104subjects.h5` on your own computer. Restart this same command after a connection interruption. When all 104 subjects are verified, upload the resulting HDF5 file to a **private Kaggle Dataset** and attach it using **Add Input**. Set `ATTACHED_CACHE_PATH` above if automatic discovery fails. The Notebook refuses to start without a complete, validated H5 file. Training loads individual EEG batches lazily; it does not load the full EEG cohort into RAM.


In [ ]:
import os, subprocess, sys
from pathlib import Path

# Prefer the preprocessed, portable HDF5 file downloaded once on your computer.
# Upload it as a PRIVATE Kaggle Dataset and attach using Add Input.
if ATTACHED_CACHE_PATH:
    cache = Path(ATTACHED_CACHE_PATH)
else:
    search_base = Path("/kaggle/input")
    attached = sorted(search_base.rglob("physionet_mi_104subjects.h5")) if search_base.is_dir() else []
    if len(attached) > 1:
        raise RuntimeError("Multiple EEG HDF5 files attached; specify ATTACHED_CACHE_PATH.")
    cache = attached[0] if attached else None
if cache is None or not cache.is_file():
    raise FileNotFoundError(
        "No verified HDF5 cache attached. Download it once on your computer using "
        "scripts/download_physionet_local.py, upload to a PRIVATE Kaggle Dataset, "
        "attach it under Add Input, and set ATTACHED_CACHE_PATH if necessary. "
        "Bulk downloading in Kaggle is intentionally disabled to avoid session crashes."
    )
if cache.suffix.lower() not in (".h5", ".hdf5"):
    raise ValueError("Expected the resumable schema-3 .h5 EEG cache, not a legacy .npz.")
print("EEG cache:", cache, "exists:", cache.is_file())
training_env = os.environ.copy()
training_env.update({
    "PHYSIONET_CACHE_PATH": str(cache),
    "PYTHONUNBUFFERED": "1",
    "MNE_DATA": str(WORK / "mne_data"),
    "MNE_DATASETS_EEGBCI_PATH": str(WORK / "mne_data"),
    "HF_HUB_DISABLE_TELEMETRY": "1",
    "MOABB_DOWNLOAD_PROVIDER": "upstream",  # PhysioNet is not served via NEMAR sourcedata
    "MPLBACKEND": "Agg",  # prevent notebook inline backend leaking into py311 subprocesses
})
Path(training_env["MNE_DATA"]).mkdir(parents=True, exist_ok=True)
# Do not enable ALLOW_UNVERIFIED_EEG_CACHE, even if an old cache is attached.
training_env.pop("ALLOW_UNVERIFIED_EEG_CACHE", None)

# Build/cache full public EEG data once (or validate attached cache).
# This subprocess releases its RAM before the training subprocess starts.
prepare_code = """from src.config import load_training_config
from src.train import load_dataset
from moabb.datasets import PhysionetMI
cfg = load_training_config('configs/baseline.yaml')
X, y, subjects = load_dataset(cfg)
expected_subjects = set(PhysionetMI().subject_list) - set(cfg.excluded_subjects)
observed = set(map(int, subjects))
if observed != expected_subjects:
    raise RuntimeError('Cache subject set does not match the full eligible PhysioNet cohort')
if X.shape[1:] != (64, 400):
    raise RuntimeError('EEG shape is incompatible with trained model')
print('FULL COHORT VERIFIED:', len(observed), 'subjects', len(y), 'EEG epochs', X.shape)
"""
log_path = RUN_DIR / "prepare_dataset.log"
with log_path.open("w", encoding="utf-8") as log:
    p = subprocess.Popen([PYTHON, "-u", "-c", prepare_code], cwd=SOURCE_DIR,
                         env=training_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
        log.write(line)
    status = p.wait()
if status:
    raise RuntimeError(f"Dataset preparation failed (exit {status}). See {log_path}.")
print("Verified private HDF5 full cohort; EEG batches will be loaded lazily during training.")


## 5. Configure the single complete experiment

**One run = one model initialization + one seed.** All runs use the same `SPLIT_SEED=42`. This session will not automatically start the remaining seeds. The config and later logs are stored under a run-specific output folder and can be retained using Kaggle Save Version.


In [ ]:
import json, hashlib, subprocess
from pathlib import Path

source_config = SOURCE_DIR / ("configs/pretrained_labram.yaml" if EXPERIMENT == "pretrained"
                              else "configs/baseline.yaml")
# YAML belongs to the pinned Python 3.11 environment; kernel Python 3.13
# only handles JSON and subprocess orchestration.
parse_yaml = "import yaml,json,sys; print(json.dumps(yaml.safe_load(open(sys.argv[1], encoding='utf-8'))))"
cfg = json.loads(subprocess.check_output([PYTHON, "-c", parse_yaml, str(source_config)], text=True))
cfg.update({
    "seed": SEED,
    "split_seed": SPLIT_SEED,
    "batch_size": BATCH_SIZE,
    "epochs": MAX_EPOCHS,
    "patience": EARLY_STOPPING_PATIENCE,
    "checkpoint_path": str(RUN_DIR / "model.pt"),
})
assert cfg["model_init"] == EXPERIMENT
assert cfg["split_seed"] == 42
if EXPERIMENT == "pretrained":
    assert cfg["pretrained_repo_id"] == "braindecode/labram-pretrained"
    assert cfg["pretrained_revision"] == "0563b6c626e7b40d9a36653b763715db94d945d7"
else:
    assert cfg["pretrained_repo_id"] is None and cfg["pretrained_revision"] is None

config_path = RUN_DIR / "experiment.yaml"
write_yaml = "import yaml,json,sys; yaml.safe_dump(json.load(sys.stdin),open(sys.argv[1],'w', encoding='utf-8'),sort_keys=True)"
subprocess.run([PYTHON, "-c", write_yaml, str(config_path)],
               input=json.dumps(cfg), text=True, check=True)
# Validate in the same interpreter used by training.
validate = """import sys
from src.config import load_training_config
c = load_training_config(sys.argv[1])
print('Validated:', c.model_init, 'seed', c.seed, 'split_seed', c.split_seed)
"""
subprocess.run([PYTHON, "-c", validate, str(config_path)], cwd=SOURCE_DIR, check=True)

meta_path = RUN_DIR / "model.json"
checkpoint = RUN_DIR / "model.pt"
if checkpoint.exists() and meta_path.exists():
    old = json.loads(meta_path.read_text(encoding="utf-8"))
    if old.get("status") == "finished" and old.get("configuration") == cfg:
        print("Matching prior completed run detected; safe to reuse this checkpoint.")
    else:
        raise RuntimeError("Existing checkpoint has no matching completed metadata; use a new RUN_LABEL.")
elif checkpoint.exists():
    raise RuntimeError("Incomplete prior checkpoint found. Choose a new RUN_LABEL; resume is not implemented.")


## 6. Execute full-data EEG training

Training uses *all eligible subjects* with a subject-disjoint Train/Validation/Test split. The best checkpoint is selected on **validation Macro-F1**; test subjects are evaluated only after selection. Training output is streamed to `train.log`. On GPU out-of-memory, rerun from a clean run folder with smaller `BATCH_SIZE` (8 or 4).


In [ ]:
import os, subprocess, sys, json
from pathlib import Path

skip_completed = False
if checkpoint.is_file() and meta_path.is_file():
    old = json.loads(meta_path.read_text(encoding="utf-8"))
    skip_completed = (old.get("status") == "finished")

if skip_completed:
    print("Using existing completed run:", RUN_DIR)
else:
    cmd = [PYTHON, "-u", "-m", "src.train", "--config", str(config_path)]
    print("Running:", " ".join(cmd))
    log_path = RUN_DIR / "train.log"
    with log_path.open("w", encoding="utf-8") as log:
        p = subprocess.Popen(cmd, cwd=SOURCE_DIR, env=training_env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="")
            log.write(line)
        returncode = p.wait()
    if returncode != 0:
        print("TRAINING FAILED. Last 5000 log characters:")
        print(log_path.read_text(encoding="utf-8", errors="replace")[-5000:])
        raise RuntimeError(f"Real-data training failed with exit {returncode}; see {log_path}")

prediction_file = RUN_DIR / "heldout_predictions.npz"
assert checkpoint.is_file() and meta_path.is_file() and prediction_file.is_file(), "Expected training artifacts missing."
metadata = json.loads(meta_path.read_text(encoding="utf-8"))
assert metadata.get("status") == "finished"
assert metadata["configuration"]["seed"] == SEED and metadata["configuration"]["split_seed"] == SPLIT_SEED
train_sub = set(metadata["train_subjects"])
val_sub = set(metadata["validation_subjects"])
test_sub = set(metadata["test_subjects"])
assert not (train_sub & val_sub or train_sub & test_sub or val_sub & test_sub), "Subject overlap detected."
assert (len(train_sub), len(val_sub), len(test_sub)) == (75,14,15), "Unexpected cohort split size."
print("Full training complete. Test accuracy:", metadata["test_accuracy"],
      "Macro-F1:", metadata["test_macro_f1"])
print("Checkpoint SHA256:", metadata["checkpoint_sha256"])


## 7. Generate genuine scientific figures and participant-aware intervals

All results below are computed from the **saved held-out probabilities**; nothing is manually inserted or simulated. Confidence intervals resample **participants**, not correlated within-participant EEG epochs.


In [ ]:
import subprocess, json, csv, re
from pathlib import Path

EVAL_DIR = RUN_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
commands = [
    [PYTHON, "-m", "scripts.build_benchmark", "--predictions", str(prediction_file),
     "--output-dir", str(EVAL_DIR)],
    [PYTHON, "-m", "scripts.subject_confidence", "--predictions", str(prediction_file),
     "--output", str(EVAL_DIR / "subject_ci.json"), "--n-bootstraps", "1000", "--seed", "42"],
]
for command in commands:
    subprocess.run(command, cwd=SOURCE_DIR, env=training_env, check=True)

# Extract actual per-epoch metrics. Use no synthetic values.
training_log = (RUN_DIR / "train.log").read_text(encoding="utf-8")
pattern = re.compile(
    r"Epoch\s+(\d+)\s*\|\s*train_loss=([0-9.eE+-]+)\s+acc=([0-9.eE+-]+)\s+f1=([0-9.eE+-]+)"
    r"\s*\|\s*val_loss=([0-9.eE+-]+)\s+acc=([0-9.eE+-]+)\s+f1=([0-9.eE+-]+)"
)
history = [tuple(float(x) for x in row) for row in pattern.findall(training_log)]
if not history:
    raise RuntimeError("No parseable real epoch logs; refusing to fabricate learning curves.")
history_file = EVAL_DIR / "training_history.csv"
with history_file.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.writer(handle)
    writer.writerow(["epoch", "train_loss", "train_accuracy", "train_macro_f1",
                     "val_loss", "val_accuracy", "val_macro_f1"])
    writer.writerows(history)

# Plot inside Python 3.11, not the Kaggle kernel.
plot_code = """import sys,csv,matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
with open(sys.argv[1], newline='') as f: rows=list(csv.DictReader(f))
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([float(r['epoch']) for r in rows], [float(r['train_macro_f1']) for r in rows], marker='o', label='Training Macro-F1')
ax.plot([float(r['epoch']) for r in rows], [float(r['val_macro_f1']) for r in rows], marker='o', label='Validation Macro-F1')
ax.set(xlabel='Epoch', ylabel='Macro-F1', title='Actual training and validation history')
ax.legend(); fig.tight_layout(); fig.savefig(sys.argv[2], dpi=180); plt.close(fig)
"""
subprocess.run([PYTHON, "-c", plot_code, str(history_file),
                str(EVAL_DIR / "learning_curves.png")], check=True)
metrics = json.loads((EVAL_DIR / "metrics.json").read_text(encoding="utf-8"))
ci = json.loads((EVAL_DIR / "subject_ci.json").read_text(encoding="utf-8"))
assert metrics["n_test_subjects"] == 15 and ci["n_heldout_subjects"] == 15
print("Held-out metrics:", {k: metrics[k] for k in
      ("accuracy", "macro_f1", "balanced_accuracy", "ece_10_bins")})
print("Subject-cluster 95% CI:", json.dumps({"accuracy": ci["accuracy"],
                                            "macro_f1": ci["macro_f1"]}, indent=2))
from IPython.display import display, Image
for figure in ("confusion_matrix.png", "reliability_diagram.png", "learning_curves.png"):
    display(Image(filename=str(EVAL_DIR / figure), width=560))


## 8. Archive verifiable evidence (excluding EEG and weights)

You can download the **small, shareable results ZIP** and send it here for independent analysis. It intentionally excludes EEG cache and model weights. The full model checkpoint is available separately in Kaggle Outputs. Save Version is recommended so the run survives after the session ends.

Never add this Notebook's caches, patient-level predictions or model checkpoints to a public GitHub commit.


In [ ]:
import json, hashlib, os, subprocess, sys, zipfile
from pathlib import Path

freeze_file = RUN_DIR / "pip-freeze.txt"
freeze_file.write_text(subprocess.check_output([PYTHON, "-m", "pip", "freeze"], text=True), encoding="utf-8")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = {
    "project": "NeuroDecode-LaBraM",
    "source_commit": actual_sha,
    "model_init": EXPERIMENT,
    "model_seed": SEED,
    "split_seed": SPLIT_SEED,
    "model_checkpoint_sha256": sha256_file(checkpoint),
    "prediction_file_sha256": sha256_file(prediction_file),
    "full_cohort_subjects": len(set(metadata["train_subjects"] + metadata["validation_subjects"] + metadata["test_subjects"])),
    "test_subjects": len(metadata["test_subjects"]),
    "accuracy": metrics["accuracy"],
    "macro_f1": metrics["macro_f1"],
    "bootstrap_method": ci["method"],
    "cache_path": str(cache),
    "cache_sha256": None,  # No misleading provenance claim; large cache may be separately hashed.
    "checkpoint_is_distributed": False,
    "results_are_full_training": True,
}
manifest_file = RUN_DIR / "run-manifest.json"
manifest_file.write_text(json.dumps(manifest, indent=2, ensure_ascii=False)+"\n", encoding="utf-8")

bundle = OUTPUT_ROOT / f"results_only_{EXPERIMENT}_seed_{SEED}{'_' + RUN_LABEL if RUN_LABEL else ''}.zip"
allowed = [config_path, meta_path, manifest_file, freeze_file, RUN_DIR/"train.log", RUN_DIR/"unit_tests.log",
           RUN_DIR/"prepare_dataset.log", RUN_DIR/"installation.log"]
allowed += sorted(EVAL_DIR.glob("*"))
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for file in allowed:
        if file.is_file():
            archive.write(file, arcname=str(file.relative_to(RUN_DIR)))
print("\nREADY. Download/share this result ZIP (without raw EEG, weight files or subject prediction NPZ):", bundle)
print("Saved separate model checkpoint:", checkpoint)
print("Saved validated local cache:", cache)
print("To complete the controlled study, repeat with:")
for model in ["pretrained", "scratch"]:
    for seed in [42, 123, 2025]:
        print(" -", model, seed)
print("Keep SPLIT_SEED=42 and the same SOURCE_COMMIT in every run.")


## After the first run: what to do next

1. Download the `results_only_pretrained_seed_42.zip` (or corresponding model/seed name) from Kaggle Outputs and send it back for review.
2. **Save Version** and preserve the validated cache as a **private** Kaggle Dataset or reuse a permitted attached cache in subsequent sessions. If `ATTACHED_CACHE_PATH` is set, data will not be re-downloaded; keep Kaggle notebook/output visibility **private** while the cache is present.
3. Repeat one run at a time for the remaining combinations: `pretrained × (42, 123, 2025)` and `scratch × (42, 123, 2025)`. Keep `SPLIT_SEED=42` and `SOURCE_COMMIT` fixed. It is acceptable to run on different GPU sessions if the rest of the protocol stays identical.
4. **Important limitation:** Resuming partway through a failed epoch is **not yet implemented** in `src.train`. A failed run must restart from a clean run-specific output directory. Keep logs for troubleshooting.
5. For Kaggle GPU out-of-memory, try `BATCH_SIZE=8` or `4`. Record any changes; a fair comparison uses the same effective training protocol for both conditions.
6. Full multi-seed comparison, any claims of improvement, and publication-grade reporting require all six successful runs. A first successful run is only one empirical observation.

**Research ethics:** The dataset is publicly available but must be used according to its own license. Do not distribute subject-level recordings or caches by accident. EEG model predictions are research results, not clinical diagnosis or a substitute for medical assessment.

**Environment note:** `python 3.13` in the Kaggle Notebook kernel is expected; the actual training, evaluation and plotted artifacts run with the managed **Python 3.11 environment** at `/kaggle/working/neurodecode_python311`. Do not reinstall Torch into the Notebook kernel.
